In [ ]:
import os
import pandas as pd
import numpy as np
import ast
import random

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)


In [ ]:
import os
os.environ["TORCH_DYNAMO_DISABLE"] = "1"

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --------- Data Loading -----------
def load_eye_tracking_data(data_dir):
    """
    Loads all eye‑tracking CSV files in `data_dir`. Each CSV must contain columns:
      - 'SegmentNumber'
      - 'QoE'
    This function fills missing numeric values, replaces any string QoE with 4,
    adds a VideoIndex, and creates a UniqueID.
    """
    all_dfs = []
    file_list = sorted(f for f in os.listdir(data_dir) if f.startswith("EyeTrackingData_") and f.endswith(".csv"))
    if file_list == []:
        file_list = sorted(f for f in os.listdir(data_dir) if f.startswith("EyeQoeData_") and f.endswith(".csv"))

    for video_idx, filename in enumerate(file_list):
        file_path = os.path.join(data_dir, filename)
        df = pd.read_csv(file_path)

        # Replace any QoE value that is a string with 4.
            # If QoE column is missing, add it with default value 0
        if "QoE" not in df.columns:
            df["QoE"] = 0
       
        df["QoE"] = pd.to_numeric(df["QoE"], errors="coerce")
        df["QoE"] = df["QoE"].apply(sanitize_qoe)
        # Fill missing numeric values with the column mean.
        df.fillna(df.mean(numeric_only=True), inplace=True)
        df["VideoIndex"] = video_idx
        if "SegmentNumber" not in df.columns:
            raise ValueError("CSV file does not contain 'SegmentNumber' column.")
        df["UniqueID"] = df["SegmentNumber"].apply(lambda seg: f"video_{video_idx}_segment_{seg}")
        all_dfs.append(df)

    df_combined = pd.concat(all_dfs, ignore_index=True)
    df_combined.fillna(0, inplace=True)
    print(f"[EyeTracking] Loaded {len(file_list)} files. Combined shape: {df_combined.shape}")
    return df_combined

def sanitize_qoe(x):
    if pd.isna(x) or x < 1 or x > 5 or x != int(x):
        return random.randint(1, 5)
    return int(x)



# --------- Utility: Get Feature Columns -----------
def get_feature_cols(df):

    exclude = ["Timestamp", "SequenceNumber", "SegmentNumber", "UniqueID", "QoE",
               "DataType", "IsEyeTrackingAvailable", "HasEyeTrackingData", "EyeTrackingStatus"]
    return [col for col in df.columns if col not in exclude]

# --------- Helper: Fix Segment Length -----------
def fix_segment_length(arr, target_length=480):

    current_length = arr.shape[0]
    if current_length > target_length:
        return arr[:target_length, :]
    elif current_length < target_length:
        pad_length = target_length - current_length
        last_row = arr[-1, :].reshape(1, -1)
        pad_array = np.repeat(last_row, pad_length, axis=0)
        return np.vstack([arr, pad_array])
    else:
        return arr

# --------- Segment Processing -----------
# Cell: Updated process_segments to handle missing QoE
def process_segments(df, target_length=480):

    # If QoE column is missing, add it with default value 0
    if "QoE" not in df.columns:
        df["QoE"] = 0

    segment_data = {}
    grouped = df.groupby("UniqueID")

    for uid, group in grouped:
        # Exclude metadata columns from raw features.
        feature_cols = get_feature_cols(df)
        raw_array    = group[feature_cols].to_numpy()
        # Ensure the segment has exactly target_length rows:
        fixed_array  = fix_segment_length(raw_array, target_length=target_length)
        # Safely read QoE now guaranteed to exist
        qoe_val      = group["QoE"].iloc[0]
        video_idx    = group["VideoIndex"].iloc[0]
        seg_num      = group["SegmentNumber"].iloc[0]
        segment_data[uid] = {
            "raw":           fixed_array,
            "QoE":           qoe_val,
            "VideoIndex":    video_idx,
            "SegmentNumber": seg_num
        }

    print(f"[Process Segments] Processed {len(segment_data)} segments (all fixed to {target_length} rows).")
    return segment_data


# --------- Define Modalities -----------
modalities = {
    "eye_data": [
         "LeftEyeOriginX", "LeftEyeOriginY", "LeftEyeOriginZ",
         "RightEyeOriginX", "RightEyeOriginY", "RightEyeOriginZ",
         "LeftEyeDirectionX", "LeftEyeDirectionY", "LeftEyeDirectionZ",
         "RightEyeDirectionX", "RightEyeDirectionY", "RightEyeDirectionZ",
         "CombinedEyeOriginX", "CombinedEyeOriginY", "CombinedEyeOriginZ",
              "CombinedEyeDirectionX", "CombinedEyeDirectionY", "CombinedEyeDirectionZ",
         "LeftEyeOpenness", "RightEyeOpenness",
         "LeftEyePupilDiameter", "RightEyePupilDiameter",
         "LeftEyePupilPositionX", "LeftEyePupilPositionY",
         "RightEyePupilPositionX", "RightEyePupilPositionY"
         # Include expressions here:
        #  "EyeExpression_LEFT_BLINK", "EyeExpression_LEFT_WIDE",
        #  "EyeExpression_RIGHT_BLINK", "EyeExpression_RIGHT_WIDE",
        #  "EyeExpression_LEFT_SQUEEZE", "EyeExpression_RIGHT_SQUEEZE",
        #  "EyeExpression_LEFT_DOWN", "EyeExpression_RIGHT_DOWN",
        #  "EyeExpression_LEFT_OUT", "EyeExpression_RIGHT_IN",
        #  "EyeExpression_LEFT_IN", "EyeExpression_RIGHT_OUT",
        #  "EyeExpression_LEFT_UP", "EyeExpression_RIGHT_UP"
    ],
    # "eye_data_without_expressions": [
    #      "LeftEyeOriginX", "LeftEyeOriginY", "LeftEyeOriginZ",
    #      "RightEyeOriginX", "RightEyeOriginY", "RightEyeOriginZ",
    #      "CombinedEyeOriginX", "CombinedEyeOriginY", "CombinedEyeOriginZ",
    #      "LeftEyeDirectionX", "LeftEyeDirectionY", "LeftEyeDirectionZ",
    #      "RightEyeDirectionX", "RightEyeDirectionY", "RightEyeDirectionZ",
    #      "CombinedEyeDirectionX", "CombinedEyeDirectionY", "CombinedEyeDirectionZ",
    #      "LeftEyeOpenness", "RightEyeOpenness",
    #      "LeftEyePupilDiameter", "RightEyePupilDiameter",
    #      "LeftEyePupilPositionX", "LeftEyePupilPositionY",
    #      "RightEyePupilPositionX", "RightEyePupilPositionY"
    # ],
    "lip_data": [
         "LipExpression_Jaw_Right", "LipExpression_Jaw_Left", "LipExpression_Jaw_Forward", "LipExpression_Jaw_Open",
            "LipExpression_Mouth_Upper_Left", "LipExpression_Mouth_Upper_Right", "LipExpression_Mouth_Lower_Left", "LipExpression_Mouth_Lower_Right",
            "LipExpression_Mouth_Smile_Right", "LipExpression_Mouth_Smile_Left", "LipExpression_Mouth_Sad_Right", "LipExpression_Mouth_Sad_Left"
            
         

    ],
    "head_data": [
         "HeadPosePositionX", "HeadPosePositionY", "HeadPosePositionZ",
         "HeadPoseRotationX", "HeadPoseRotationY", "HeadPoseRotationZ"
    ]
}

def create_modality_combinations(modalities):
   
    from itertools import combinations
    keys = list(modalities.keys())
    all_combos = []
    for r in range(1, len(keys)+1):
        for combo in combinations(keys, r):
            # Skip redundant combinations.
            if "eye_data" in combo and "eye_data_without_expressions" in combo:
                continue
            all_combos.append(combo)
    return all_combos

# --------- Dataset Balancing -----------
def balance_segments(segment_data, random_state=42):
   
    from collections import defaultdict
    import random
    qoe_groups = defaultdict(list)
    for uid, data in segment_data.items():
        qoe_groups[data["QoE"]].append(uid)

    # Log distribution before balancing.
    print("\n[Balance] Distribution before balancing:")
    for qoe, uid_list in qoe_groups.items():
        print(f"  QoE = {qoe}: {len(uid_list)} segments")

    min_count = min(len(uids) for uids in qoe_groups.values()) + 10
    print(f"[Balance] Minimum segments per QoE class: {min_count}")

    random.seed(random_state)
    balanced_ids = []
    for qoe, uid_list in qoe_groups.items():
        if len(uid_list) > min_count:
            sampled_ids = random.sample(uid_list, min_count)
        else:
            sampled_ids = uid_list
        balanced_ids.extend(sampled_ids)

    balanced_data = {uid: segment_data[uid] for uid in balanced_ids}

    # Log distribution after balancing.
    new_qoe_groups = defaultdict(list)
    for uid, data in balanced_data.items():
        new_qoe_groups[data["QoE"]].append(uid)
    print("\n[Balance] Distribution after balancing:")
    for qoe, uid_list in new_qoe_groups.items():
        print(f"  QoE = {qoe}: {len(uid_list)} segments")

    print(f"[Balance] Reduced from {len(segment_data)} to {len(balanced_data)} segments.")
    return balanced_data

# --------- Run Data Loading and Processing -----------
# (Update this path to your data directory.)
data_dir = "./Data3"  # <-- update as needed
df_eye = load_eye_tracking_data(data_dir)
segment_data = process_segments(df_eye, 48)

# Log total segments and a sample of their QoE values before balancing.
print(f"\nTotal segments before balancing: {len(segment_data)}")
sample_qoe = [data["QoE"] for uid, data in list(segment_data.items())[:5]]
print("Sample QoE values before balancing:", sample_qoe)

balanced_segment_data = balance_segments(segment_data)
print(f"\nTotal balanced segments: {len(balanced_segment_data)}")

# Set global feature_cols_full to include only numeric features (exclude metadata)
feature_cols_full = get_feature_cols(df_eye)
print("\nFeature columns:", feature_cols_full)

# Create modality combinations
modality_combos = create_modality_combinations(modalities)
print("\nModality combinations:", modality_combos)


In [ ]:

def extract_modality_data(segment_data, modalities, combo):
    """
    For each segment in segment_data, select only the columns corresponding to the modalities in combo.
    Returns:
      - raw_list: list of raw arrays (each of shape (480, n_selected_features))
      - flat_data: dict mapping UniqueID to a flattened 1D vector
      - labels: dict mapping UniqueID to its QoE
      - unique_ids: list of UniqueIDs processed
    """
    raw_list = []
    flat_data = {}
    labels = {}
    unique_ids = []

    for uid, data in segment_data.items():
        # raw_arr is of shape (480, n_features_full)
        raw_arr = data["raw"]
        selected_cols = []
        for mod in combo:
            if mod in modalities:
                selected_cols.extend(modalities[mod])
        # Get indices from global feature_cols_full that are in the selected columns.
        selected_indices = [i for i, col in enumerate(feature_cols_full) if col in selected_cols]
        if not selected_indices:
            continue
        raw_selected = raw_arr[:, selected_indices]
        raw_list.append(raw_selected)
        flat_data[uid] = raw_selected.flatten()
        labels[uid] = data["QoE"]
        unique_ids.append(uid)

    return raw_list, flat_data, labels, unique_ids


In [ ]:
# Cell: Train RF on Data3 and batch‑infer QoE for Eye_train using extract_modality_data
import os
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Paths
data3_dir     = "./Data3"
eye_train_dir = "./QoE_train/Eye_train"
preds_csv     = "./Preds/preds_RF_EyeTrain.csv"
os.makedirs(os.path.dirname(preds_csv), exist_ok=True)

# --- 1) Train RF on Data3 ---
df_eye3 = load_eye_tracking_data(data3_dir)
seg3    = process_segments(df_eye3, 48)
bal3    = balance_segments(seg3)
feature_cols_full = get_feature_cols(df_eye3)
combo = ("eye_data",)

# Flatten Data3 with modality extraction
_, flat_dict3, label_dict3, _ = extract_modality_data(
    bal3, modalities, combo
)
df_flat3 = pd.DataFrame.from_dict(flat_dict3, orient="index")
df_flat3["QoE"] = pd.Series(label_dict3)

# Train RF
X3 = df_flat3.drop(columns=["QoE"])
y3 = df_flat3["QoE"]
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X3, y3)
print(f"[RF] Trained on Data3: {X3.shape[0]} samples, {X3.shape[1]} features")

# --- 2) Inference on Eye_train via extract_modality_data ---
# Load and process Eye_train
df_eye_t = load_eye_tracking_data(eye_train_dir)
seg_t    = process_segments(df_eye_t, 48)
# No balancing needed; we want every segment
_, flat_dict_t, _, _ = extract_modality_data(
    seg_t, modalities, combo
)
# flat_dict_t: UID -> feature vector (same dims as X3)

# Batch inference
results = []
uids = list(flat_dict_t.keys())
batch_size = 2000
for i in range(0, len(uids), batch_size):
    batch_ids = uids[i:i+batch_size]
    batch_mat = [flat_dict_t[uid] for uid in batch_ids]
    Xb = pd.DataFrame(batch_mat).apply(pd.to_numeric, errors="coerce").fillna(0)
    preds = rf.predict(Xb)
    results.extend(zip(batch_ids, preds))
    print(f"Inferred batch {i//batch_size+1} ({len(batch_ids)} segments)")

# --- 3) Save to CSV ---
df_out = pd.DataFrame(results, columns=["UniqueID", "PredictedQoE"])
df_out.to_csv(preds_csv, index=False)
print(f"[Done] Saved {len(df_out)} predictions to {preds_csv}")


In [ ]:
combo = ("eye_data", "lip_data", "head_data")
segmentDuration = 48
# --- Data2 Processing (Training Data) ---
data1_dir = "./QoE_train/Eye_train"
# Update as needed.
df_eye_train = load_eye_tracking_data(data1_dir)
segment_data_train = process_segments(df_eye_train, target_length=segmentDuration)
balanced_segment_data= balance_segments(segment_data_train)

In [ ]:
import os
import pandas as pd
import ast

def load_network_metrics(network_dir):
    """
    Loads network CSV files from `network_dir`. Each CSV corresponds to a video.
    Expected columns: "Segment Number", "Buffer Level", "Bitrate (kbps)",
                      "Throughput History", "Rebuffer Time"
    Returns a dict mapping UniqueID (e.g., "video_0_segment_1") to network metrics.
    """
    net_data = {}
    file_list = sorted(f for f in os.listdir(network_dir) if f.endswith(".csv"))
    for video_idx, filename in enumerate(file_list):
        print(f"Loading network file: {filename}...")
        file_path = os.path.join(network_dir, filename)
        df_net = pd.read_csv(file_path)
        df_net.rename(columns={
            "Segment Number": "SegmentNumber",
            "Buffer Level": "BufferLevel",
            "Bitrate (kbps)": "Bitrate",
            "Throughput History": "ThroughputHistory",
            "Rebuffer Time": "RebufferTime"
        }, inplace=True)
        for _, row in df_net.iterrows():
            seg_num = row["SegmentNumber"]
            uid = f"video_{video_idx}_segment_{seg_num}"
            try:
                throughput_list = ast.literal_eval(str(row.get("ThroughputHistory", "[]")))
            except:
                throughput_list = []
            net_data[uid] = {
                "BufferLevel": row.get("BufferLevel", 0.0) / 80.0,
                "ThroughputHistory": throughput_list,
                "RebufferTime": row.get("RebufferTime", 0.0),
                "Bitrate": row.get("Bitrate", 0.0)
            }
    print(f"[NetworkData] Loaded {len(file_list)} network files. Total segments: {len(net_data)}")
    return net_data

def merge_data_for_segments(balanced_eye, net_data):
 
    merged = {}
    for uid, eye_data in balanced_eye.items():
        if uid in net_data:
            merged[uid] = {
                "raw": eye_data["raw"],
                "qoe": eye_data["QoE"],
                "network": net_data[uid]
            }
        else:
            print(f"[Merge] Warning: No network data for {uid}. Skipping.")
    print(f"[Merge] Merged data for {len(merged)} segments.")
    return {("eye_data",): merged}

network_dir = "./QoE_train/Net_train"  # Update as needed
net_data = load_network_metrics(network_dir)
merged_dict = merge_data_for_segments(balanced_segment_data, net_data)


In [ ]:

def process_segments_for_modality(df, modality_key):
 
    if modality_key not in modalities:
        raise ValueError(f"Modality '{modality_key}' not found in modalities dictionary.")
    desired = modalities[modality_key]
    # Only use columns that exist in the DataFrame.
    cols = [col for col in desired if col in df.columns]
    print(f"[{modality_key}] Selected {len(cols)} columns.")

    segment_dict = {}
    grouped = df.groupby("UniqueID")
    for uid, group in grouped:
        # Extract only the columns for this modality.
        arr = group[cols].to_numpy()
        # Fix each segment to exactly 480 rows.
        fixed_arr = fix_segment_length(arr, target_length=segmentDuration)
        qoe_val = group["QoE"].iloc[0]
        vid = group["VideoIndex"].iloc[0]
        seg_num = group["SegmentNumber"].iloc[0]
        segment_dict[uid] = {"raw": fixed_arr, "QoE": qoe_val, "VideoIndex": vid, "SegmentNumber": seg_num}
    print(f"[{modality_key}] Processed {len(segment_dict)} segments ")
    return segment_dict

def merge_modality_segments(balanced_data, modality_segment_data, net_data):
 
    merged = {}
    skipped = 0
    for uid in balanced_data.keys():
        if uid in modality_segment_data and uid in net_data:
            merged[uid] = {
                "raw": modality_segment_data[uid]["raw"],
                "qoe": modality_segment_data[uid]["QoE"],
                "network": net_data[uid]
            }
        else:
            skipped += 1
    print(f"[Merge] For modality, merged {len(merged)} segments; skipped {skipped} segments due to missing data.")
    return merged

# Process each modality separately using the full DataFrame 'df_eye_train'
eye_segments = process_segments_for_modality(df_eye_train, "eye_data")
lip_segments = process_segments_for_modality(df_eye_train, "lip_data")
head_segments = process_segments_for_modality(df_eye_train, "head_data")

# Merge with network data – filter by balanced_segment_data so that only segments in the balanced set are kept.
merged_eye  = merge_modality_segments(balanced_segment_data, eye_segments, net_data)
merged_lip  = merge_modality_segments(balanced_segment_data, lip_segments, net_data)
merged_head = merge_modality_segments(balanced_segment_data, head_segments, net_data)

# Optionally, pack into one dictionary:
merged_dict = {
    ("eye_data",): merged_eye,
    ("lip_data",): merged_lip,
    ("head_data",): merged_head
}

print("\nFinal merged dictionary keys:", merged_dict.keys())


In [ ]:
# ---------------------------
# Debug Cell: Check Final Merged Dictionary for Missing Data
# ---------------------------
import pandas as pd

def debug_merged_dict(merged_dict):
    """
    Iterates over the final merged dictionary and prints, for each modality combo,
    how many segments have missing 'raw', 'qoe', or 'network' data.
    """
    for modality_combo, merged_data in merged_dict.items():
        print(f"\n[Debug] Modality combo: {modality_combo}")
        total_segments = len(merged_data)

        missing_raw = [uid for uid, data in merged_data.items() if data.get("raw") is None]
        missing_qoe = [uid for uid, data in merged_data.items() if data.get("qoe") is None]
        missing_network = [uid for uid, data in merged_data.items() if data.get("network") is None]

        print(f"Total segments: {total_segments}")
        print(f"  Segments with missing 'raw': {len(missing_raw)}")
        print(f"  Segments with missing 'qoe': {len(missing_qoe)}")
        print(f"  Segments with missing 'network': {len(missing_network)}")

        # Optionally, print sample UniqueIDs for each missing category.
        if missing_raw:
            print("  Sample segments with missing 'raw':", missing_raw[:5])
        if missing_qoe:
            print("  Sample segments with missing 'qoe':", missing_qoe[:5])
        if missing_network:
            print("  Sample segments with missing 'network':", missing_network[:5])

        # Additionally, count how many segments have complete data (none missing).
        complete = [uid for uid, data in merged_data.items()
                    if (data.get("raw") is not None and data.get("qoe") is not None and data.get("network") is not None)]
        print(f"  Segments with complete data: {len(complete)} (out of {total_segments})")

# Call the debug function on the final merged dictionary
debug_merged_dict(merged_dict)


In [ ]:
def gather_columns_for_combo(df, combo, modalities):
    """
    Gathers the columns from the DataFrame 'df' corresponding to the modalities in 'combo'.
    It returns a list of column names with duplicates removed while preserving order.

    For example, if combo = ("eye_data", "lip_data", "head_data"), it returns a list of
    columns from modalities["eye_data"] + modalities["lip_data"] + modalities["head_data"]
    that exist in df.
    """
    col_list = []
    for mod in combo:
        if mod not in modalities:
            print(f"[Warning] Modality {mod} not found in modalities dictionary; skipping.")
            continue
        col_list.extend(modalities[mod])
    # Remove duplicates while preserving order
    seen = set()
    final_cols = []
    for col in col_list:
        if col not in seen and col in df.columns:
            seen.add(col)
            final_cols.append(col)
    print(f"[Info] For combo {combo}, selected {len(final_cols)} columns.")
    return final_cols

def build_and_merge_combo(df_big, balanced_data, net_data, modalities, combo):
    """
    Builds and merges the dataset for a given modality combination.

    Steps:
      A) Determine which columns to use from the full DataFrame 'df_big' using the provided
         modality combo and the global 'modalities' dictionary.
      B) Group 'df_big' by UniqueID and, for each segment that is present in the balanced set,
         extract the required columns, fix the segment length to exactly 480 rows, and flatten
         the array. Also extract the segment's QoE.
      C) Merge this processed segment data with the network data (net_data) by matching UniqueID.

    Returns:
      A dictionary of the form:
          { combo: { UniqueID -> { "raw": flattened_array, "qoe": float, "network": network_dict } } }
    """
    # Step A: Get needed columns.
    needed_cols = gather_columns_for_combo(df_big, combo, modalities)

    # Step B: Process each segment from the full DataFrame.
    grouped = df_big.groupby("UniqueID")
    combo_dict = {}
    for uid, seg_df in grouped:
        if uid not in balanced_data:
            continue  # Only include segments that are in the balanced set.
        arr = seg_df[needed_cols].to_numpy()
        arr_fixed = fix_segment_length(arr, target_length=segmentDuration)
        qoe_val = seg_df["QoE"].iloc[0]
        combo_dict[uid] = {"raw": arr_fixed, "qoe": qoe_val}

    # Step C: Merge with network metrics.
    merged_out = {}
    skipped = 0
    for uid, data in combo_dict.items():
        if uid in net_data:
            merged_out[uid] = {
                "raw": data["raw"],
                "qoe": data["qoe"],
                "network": net_data[uid]
            }
        else:
            skipped += 1
    print(f"[Merge:{combo}] Built {len(merged_out)} segments; skipped {skipped} segments (missing network data).")
    return {combo: merged_out}


In [ ]:
merged_dict = {}

# Build entry for ("eye_data",)
merged_dict.update(
    build_and_merge_combo(
        df_eye_train,
        balanced_segment_data,
        net_data,
        modalities,
        combo=("eye_data",)
    )
)

# Build entry for ("lip_data",)
merged_dict.update(
    build_and_merge_combo(
        df_eye_train,
        balanced_segment_data,
        net_data,
        modalities,
        combo=("lip_data",)
    )
)

# Build entry for ("head_data",)
merged_dict.update(
    build_and_merge_combo(
        df_eye_train,
        balanced_segment_data,
        net_data,
        modalities,
        combo=("head_data",)
    )
)

# Now merged_dict.keys() => dict_keys([("eye_data",), ("lip_data",), ("head_data",)])


In [ ]:
def combine_modalities_into_final(merged_dict, balanced_segment_data):
    """
    1) For each segment in 'balanced_segment_data', gather 'raw' arrays
       from merged_dict[("eye_data",)], merged_dict[("lip_data",)], merged_dict[("head_data",)].
    2) Build final_dict[uid] = {
         "eye": (480,40),
         "lip": (480,37),
         "head":(480,6),
         "qoe": (maybe from eye? or lip?),
         "network": ...
       }
    Returns final_dict.
    """
    final_dict = {}

    # Extract sub-dicts from merged_dict
    eye_sub  = merged_dict.get(("eye_data",),  {})
    lip_sub  = merged_dict.get(("lip_data",),  {})
    head_sub = merged_dict.get(("head_data",), {})

    for uid in balanced_segment_data.keys():
        # We create a new entry
        final_dict[uid] = {}

        # Eye
        if uid in eye_sub:
            final_dict[uid]["eye"]      = eye_sub[uid]["raw"]  # shape => (480,40)
            final_dict[uid]["qoe"]      = eye_sub[uid]["qoe"]
            final_dict[uid]["network"]  = eye_sub[uid]["network"]
        else:
            final_dict[uid]["eye"]      = None

        # Lip
        if uid in lip_sub:
            final_dict[uid]["lip"]      = lip_sub[uid]["raw"]  # shape => (480,37)
            # Optionally unify qoe/network if needed, or check if they match
        else:
            final_dict[uid]["lip"]      = None

        # Head
        if uid in head_sub:
            final_dict[uid]["head"]     = head_sub[uid]["raw"] # shape => (480,6)
            # same note re: qoe/network
        else:
            final_dict[uid]["head"]     = None

        # If you want to ensure consistent QOE/net across all, you can compare
        # final_dict[uid]["qoe"] with lip_sub[uid]["qoe"], etc., if needed.

    return final_dict


In [ ]:
# Diagnostic: Compare keys between balanced_segment_data and each modality's processed data.

balanced_ids = set(balanced_segment_data.keys())
eye_ids = set(eye_segments.keys())
lip_ids = set(lip_segments.keys())
head_ids = set(head_segments.keys())

print("Total keys in balanced_segment_data:", len(balanced_ids))
print("Total keys in eye_segments:", len(eye_ids))
print("Total keys in lip_segments:", len(lip_ids))
print("Total keys in head_segments:", len(head_ids))

missing_in_eye = balanced_ids - eye_ids
missing_in_lip = balanced_ids - lip_ids
missing_in_head = balanced_ids - head_ids

print("\nNumber of balanced keys missing in eye_segments:", len(missing_in_eye))
print("Sample missing (eye):", list(missing_in_eye)[:5])
print("\nNumber of balanced keys missing in lip_segments:", len(missing_in_lip))
print("Sample missing (lip):", list(missing_in_lip)[:5])
print("\nNumber of balanced keys missing in head_segments:", len(missing_in_head))
print("Sample missing (head):", list(missing_in_head)[:5])


In [ ]:
# Compare the UniqueIDs in balanced_segment_data vs. each merged modality sub-dictionary

# Get the set of UniqueIDs from balanced_segment_data.
balanced_keys = set(balanced_segment_data.keys())

# For each modality, get the set of keys from the merged dictionary.
eye_keys = set(merged_dict.get(("eye_data",), {}).keys())
lip_keys = set(merged_dict.get(("lip_data",), {}).keys())
head_keys = set(merged_dict.get(("head_data",), {}).keys())

print("=== Key Comparison ===")
print(f"Total Balanced Keys: {len(balanced_keys)}")

print("\n--- Eye Data ---")
print(f"Merged Eye Keys Count: {len(eye_keys)}")
missing_in_eye = balanced_keys - eye_keys
extra_in_eye = eye_keys - balanced_keys
print(f"Balanced keys missing in Eye: {len(missing_in_eye)}")
if missing_in_eye:
    print("Sample missing (eye):", list(missing_in_eye)[:5])
print(f"Extra keys in Eye (not in balanced): {len(extra_in_eye)}")
if extra_in_eye:
    print("Sample extra (eye):", list(extra_in_eye)[:5])

print("\n--- Lip Data ---")
print(f"Merged Lip Keys Count: {len(lip_keys)}")
missing_in_lip = balanced_keys - lip_keys
extra_in_lip = lip_keys - balanced_keys
print(f"Balanced keys missing in Lip: {len(missing_in_lip)}")
if missing_in_lip:
    print("Sample missing (lip):", list(missing_in_lip)[:5])
print(f"Extra keys in Lip (not in balanced): {len(extra_in_lip)}")
if extra_in_lip:
    print("Sample extra (lip):", list(extra_in_lip)[:5])

print("\n--- Head Data ---")
print(f"Merged Head Keys Count: {len(head_keys)}")
missing_in_head = balanced_keys - head_keys
extra_in_head = head_keys - balanced_keys
print(f"Balanced keys missing in Head: {len(missing_in_head)}")
if missing_in_head:
    print("Sample missing (head):", list(missing_in_head)[:5])
print(f"Extra keys in Head (not in balanced): {len(extra_in_head)}")
if extra_in_head:
    print("Sample extra (head):", list(extra_in_head)[:5])


In [ ]:
def combine_modalities_into_final_debug(merged_dict, balanced_segment_data):
    """
    Combine modalities using balanced_segment_data as the base.
    Logs for each UniqueID whether data from 'eye', 'lip', and 'head' exist.
    Returns final_dict.
    """
    final_dict = {}

    # Extract sub-dicts from merged_dict
    eye_sub  = merged_dict.get(("eye_data",),  {})
    lip_sub  = merged_dict.get(("lip_data",),  {})
    head_sub = merged_dict.get(("head_data",), {})

    missing_counts = {"eye": 0, "lip": 0, "head": 0}
    for uid in balanced_segment_data.keys():
        final_dict[uid] = {}

        # Eye
        if uid in eye_sub:
            final_dict[uid]["eye"]      = eye_sub[uid]["raw"]
            final_dict[uid]["qoe"]      = eye_sub[uid]["qoe"]
            final_dict[uid]["network"]  = eye_sub[uid]["network"]
        else:
            final_dict[uid]["eye"]      = None
            missing_counts["eye"] += 1

        # Lip
        if uid in lip_sub:
            final_dict[uid]["lip"]      = lip_sub[uid]["raw"]
        else:
            final_dict[uid]["lip"]      = None
            missing_counts["lip"] += 1

        # Head
        if uid in head_sub:
            final_dict[uid]["head"]     = head_sub[uid]["raw"]
        else:
            final_dict[uid]["head"]     = None
            missing_counts["head"] += 1

    print("\n[combine_modalities_into_final_debug] Summary:")
    print(f"  Total segments (balanced): {len(balanced_segment_data)}")
    print(f"  Segments missing eye data: {missing_counts['eye']}")
    print(f"  Segments missing lip data: {missing_counts['lip']}")
    print(f"  Segments missing head data: {missing_counts['head']}")
    return final_dict

# Use the debug version instead:
final_dict_debug = combine_modalities_into_final_debug(merged_dict, balanced_segment_data)
print(f"\nFinal combined dictionary (debug) contains {len(final_dict_debug)} segments.")


In [ ]:
def inspect_final_missing(final_dict):
    missing_entries = {uid: data for uid, data in final_dict.items()
                       if data.get("eye") is None or data.get("lip") is None or data.get("head") is None or data.get("qoe") is None or data.get("network") is None}
    print(f"Total segments with missing data: {len(missing_entries)}")
    for uid, data in list(missing_entries.items())[:10]:
        print(f"Segment {uid} => {data}")

inspect_final_missing(final_dict_debug)


In [ ]:
final_dict = final_dict_debug
merged_dict = final_dict


In [ ]:
def debug_final_dict(final_dict):
    total = len(final_dict)
    missing_eye = 0
    missing_lip = 0
    missing_head = 0
    missing_qoe = 0
    missing_network = 0

    # Collect sample entries where any data is missing.
    samples = []

    for uid, data in final_dict.items():
        if data.get("eye") is None:
            missing_eye += 1
        if data.get("lip") is None:
            missing_lip += 1
        if data.get("head") is None:
            missing_head += 1
        if data.get("qoe") is None:
            missing_qoe += 1
        if data.get("network") is None:
            missing_network += 1

        # If any field is missing, record this UID and its data.
        if (data.get("eye") is None or data.get("lip") is None or
            data.get("head") is None or data.get("qoe") is None or
            data.get("network") is None):
            samples.append((uid, data))

    print("====== Final Dictionary Debug Summary ======")
    print(f"Total segments: {total}")
    print(f"Segments missing 'eye' data: {missing_eye}")
    print(f"Segments missing 'lip' data: {missing_lip}")
    print(f"Segments missing 'head' data: {missing_head}")
    print(f"Segments missing 'qoe': {missing_qoe}")
    print(f"Segments missing 'network': {missing_network}")

    if samples:
        print("\nSample segments with missing data:")
        for uid, data in samples[:5]:
            print(f"Segment {uid} => {data}")
    else:
        print("\nAll segments have complete data.")

# Run the debug function on your final dictionary.
debug_final_dict(final_dict)


In [ ]:
def debug_df_eye(df_eye, label="df_eye"):
    """
    Prints basic info about the DataFrame df_eye.
    """
    if df_eye is None:
        print(f"[DEBUG] {label} is None.")
        return
    if df_eye.empty:
        print(f"[DEBUG] {label} is an empty DataFrame.")
        return

    print(f"[DEBUG] {label}: shape={df_eye.shape}")
    print(f"  Columns: {list(df_eye.columns)}")
    # Print a small sample of rows
    print(df_eye.head(3))

def debug_segment_data(segment_data, label="segment_data", max_print=3):
    """
    Prints how many segments, logs a sample of keys, and prints QoE for a few segments.
    segment_data => { uid-> { 'raw':..., 'QoE':..., ... } }
    """
    if not segment_data:
        print(f"[DEBUG] {label} is empty or None!")
        return
    print(f"[DEBUG] {label} => {len(segment_data)} entries. Sample keys: {list(segment_data.keys())[:max_print]}")
    # Print sample detail
    count = 0
    for uid in list(segment_data.keys())[:max_print]:
        info = segment_data[uid]
        print(f"  -> {uid}: QoE={info.get('QoE', None)}, raw.shape={info['raw'].shape if 'raw' in info else None}")
        count += 1

def debug_net_data(net_data, label="net_data", max_print=3):
    """
    Prints how many UIDs in net_data, logs a sample, checks for BufferLevel or ThroughputHistory.
    net_data => { uid-> { 'BufferLevel':..., 'ThroughputHistory':..., ... } }
    """
    if not net_data:
        print(f"[DEBUG] {label} is empty or None!")
        return
    print(f"[DEBUG] {label} => {len(net_data)} entries. Sample keys: {list(net_data.keys())[:max_print]}")
    for uid in list(net_data.keys())[:max_print]:
        info = net_data[uid]
        print(f"  -> {uid}: BufferLevel={info.get('BufferLevel', None)}, "
              f"ThroughputHistory len={len(info.get('ThroughputHistory', []))}")

def debug_merged_dict(merged_dict, label="merged_dict", max_print=3):
    """
    If merged_dict is { ("eye_data",): { uid-> {...} }, ... },
    we print the keys (the combos) and sample from each sub-dict.
    """
    if not merged_dict:
        print(f"[DEBUG] {label} is empty or None!")
        return
    print(f"[DEBUG] {label} has combos => {list(merged_dict.keys())}")
    for combo_key in merged_dict.keys():
        sub_dict = merged_dict[combo_key]
        print(f"  Combo {combo_key}: {len(sub_dict)} segments. Sample:")
        for uid in list(sub_dict.keys())[:max_print]:
            info = sub_dict[uid]
            print(f"    -> {uid}: qoe={info.get('qoe', None)}, "
                  f"raw.shape={info['raw'].shape if 'raw' in info else None}, "
                  f"network keys={list(info['network'].keys()) if info.get('network') else None}")


In [ ]:
def count_missing(final_dict):
    """
    Iterates over the final merged dictionary and counts how many segments
    have None for each key: 'eye', 'lip', 'head', 'qoe', and 'network'.
    Also counts how many segments have all values as None.
    """
    # Initialize counts for each key and for segments missing everything.
    counts = {"eye": 0, "lip": 0, "head": 0, "qoe": 0, "network": 0, "all": 0}
    total_segments = len(final_dict)
    missing_details = []  # To record segments with all missing

    for seg_id, seg_data in final_dict.items():
        # Track missing keys for this segment.
        missing_keys = []
        for key in ["eye", "lip", "head", "qoe", "network"]:
            if seg_data.get(key) is None:
                counts[key] += 1
                missing_keys.append(key)
        # If all keys are missing:
        if len(missing_keys) == 5:
            counts["all"] += 1
            missing_details.append(seg_id)

    print(f"Total segments in final merged dictionary: {total_segments}\n")
    print("Missing counts per key:")
    for key, count in counts.items():
        print(f"  {key}: {count}")

    print("\nSegments with all data missing:")
    if missing_details:
        for seg in missing_details:
            print(f"  {seg}")
    else:
        print("  None")

    # Optionally, you can save the missing segments details to a CSV.
    missing_df = pd.DataFrame(list(final_dict.items()), columns=["SegmentID", "Data"])
    # Filter rows where all keys are None.
    def all_none(data):
        return all(data.get(k) is None for k in ["eye", "lip", "head", "qoe", "network"])

    missing_df = missing_df[missing_df["Data"].apply(all_none)]
    output_csv = "missing_segments_summary.csv"
    missing_df.to_csv(output_csv, index=False)
    print(f"\nMissing segment details saved to '{output_csv}'.")

# Assuming your final merged dictionary is named final_dict
# (for example, it might be merged_dict if you merged modalities into one final dict)
count_missing(final_dict)


In [ ]:
# ----- Cell: Process Merged Network Data -----
def process_network_data(merged_dict):
    """
    For each segment in merged_dict, update the network dictionary as follows:
      - Convert all throughput values (assumed to be in bytes/second) to Mbps.
        (Mbps = (bytes/sec * 8) / 1e6)
      - Truncate the BufferLevel to a maximum of 6 seconds.
    """
    for uid, info in merged_dict.items():
        net = info.get("network", {})
        # Convert ThroughputHistory to Mbps if it exists.
        if "ThroughputHistory" in net:
            # Convert each sample: Mbps = (bytes/sec * 8)/1e6
            net["ThroughputHistory"] = [ (val * 8 / 1e6) for val in net["ThroughputHistory"] ]
        # Truncate BufferLevel to 6 seconds maximum.
        if "BufferLevel" in net:
            net["BufferLevel"] = min(net["BufferLevel"], 6.0)
    print("[INFO] Processed merged network data: ThroughputHistory converted to Mbps and BufferLevel truncated to 6 sec.")

# Call the processing function before any model-related code.
process_network_data(merged_dict)


In [ ]:
import numpy as np

# Define the bitrates (actions)
bitrates = [
    2000, 6000, 9500, 15000,30000, 85000
]

def calculate_reward(selected_action,
                     throughput_history,
                     buffer_level,
                     rebuffer_time,
                     subjective_qoe,
                     previous_action=None,
                     alpha=0.6,
                     buffer_threshold=80.0
                     ):
    print("bitrate", selected_action, " rebuffer_time", rebuffer_time, " subjective_qoe", subjective_qoe)
    bitrate = selected_action
    max_bitrate = max(bitrates)
    if previous_action is None:
        previous_bitrate = bitrate
    else:
        previous_bitrate = previous_action
        previous_action = previous_action
    q_bitrate = bitrate / max_bitrate
    if rebuffer_time > 1.0:
        rebuffer_time = rebuffer_time/10
    rebuffer_penalty = 1.3 * rebuffer_time
    smoothness_penalty = abs(bitrate - previous_bitrate) / max_bitrate
    
    objective_qoe = q_bitrate - rebuffer_penalty - smoothness_penalty
   
    reward = alpha * (subjective_qoe / 5) + (1 - alpha) * objective_qoe
    

    if np.isnan(reward):
        print(f"[DEBUG] NaN reward computed: bitrate {bitrate}, subjective_qoe {subjective_qoe}, objective_qoe {objective_qoe}")
    return reward

def predict_subjective_qoe(segment_id, baseline_preds,
                           rf_model, rf_labelenc,
                           raw_array=None):
    import numpy as np

    # 1) Try baseline_preds
    for model_name in ["RF","CNN","LSTM"]:
        if model_name in baseline_preds:
            val = baseline_preds[model_name].get(segment_id, None)
            if val is not None:
                # Found a baseline pred => return it
                return float(val)

    
    if raw_array is not None:
        # Flatten => shape (1, n_flat)
        flat_data = raw_array.flatten().reshape(1, -1)
        df_test = pd.DataFrame(flat_data)
        
        df_test = df_test.apply(pd.to_numeric, errors='coerce').fillna(df_test.mean())
        pred_enc = rf_model.predict(df_test)[0]  # single int label
        pred_qoe = rf_labelenc.inverse_transform([pred_enc])[0]
        return float(pred_qoe)

    
    print(f"[Warning] No raw_array to do on-the-fly RF inference => returning 0.0 for {segment_id}")
    return 0.0



In [ ]:
# ------------------------- Cell 3 -------------------------
def extract_modality_data(segment_data, modalities, combo, feature_cols_full):
    """
    For each segment in segment_data, build:
      - raw_list: list of np.array(480, n_features)
      - flat_data_dict: { uid-> flattened array }
      - label_dict: { uid-> QoE }
      - uid_list: list of segment IDs processed
    This requires that segment_data[uid]["raw"] has shape (480, N_full),
    and we pick columns from 'feature_cols_full' that match the modality columns in 'combo'.
    """
    raw_list = []
    flat_data_dict = {}
    label_dict = {}
    uid_list = []

    # Combine columns from the combo
    selected_cols = []
    for m in combo:
        if m in modalities:
            selected_cols.extend(modalities[m])

    # For each segment
    for uid, segdict in segment_data.items():
        arr_480 = segdict["raw"]
        # figure out which indices to take
        selected_indices = [i for i, colname in enumerate(feature_cols_full) if colname in selected_cols]
        if not selected_indices:
            continue
        arr_sel = arr_480[:, selected_indices]
        raw_list.append(arr_sel)
        flat_data_dict[uid] = arr_sel.flatten()
        label_dict[uid] = segdict["QoE"]
        uid_list.append(uid)

    return raw_list, flat_data_dict, label_dict, uid_list

def predict_subjective_qoe_on_the_fly(segment_id, baseline_preds, rf_model, rf_label_enc, raw_array=None):
    """
    If baseline_preds has an entry => return that.
    Else flatten raw_array => do inference with rf_model => decode => float
    Otherwise => 0.0 fallback.
    """
    # 1) Try baseline preds
    for model_name in ["RF","CNN","LSTM"]:
        if model_name in baseline_preds:
            if segment_id in baseline_preds[model_name]:
                return float(baseline_preds[model_name][segment_id])
    # 2) On-the-fly
    if raw_array is not None:
        flat = raw_array.flatten().reshape(1,-1)
        df_test = pd.DataFrame(flat)
        df_test = df_test.apply(pd.to_numeric, errors='coerce').fillna(df_test.mean())
        pred_enc = rf_model.predict(df_test)[0]
        pred_float = rf_label_enc.inverse_transform([pred_enc])[0]
        return float(pred_float)
    # 3) Fallback
    return 0.0


In [ ]:

        
import torch
import torch.nn as nn
import torch.nn.functional as F

FEATURE_NUM = 128

class DQNBIO(nn.Module):

    def __init__(self, action_dim=6):
        super(DQNBIO, self).__init__()
        self.action_dim = action_dim

        
        self.fc0 = nn.Linear(1, FEATURE_NUM)   
        self.fc1 = nn.Linear(1, FEATURE_NUM)   
        self.fc2 = nn.Linear(8, FEATURE_NUM)   
        self.fc3 = nn.Linear(8, FEATURE_NUM)   
        self.fc4 = nn.Linear(action_dim, FEATURE_NUM)  
        self.fc5 = nn.Linear(1, FEATURE_NUM)  

       
        self.eye_cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )
  
        self.eye_fc = nn.Linear(64 * 48 * 26, FEATURE_NUM)

        self.lip_cnn = nn.Sequential(
            nn.Conv2d(1,16,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.Conv2d(16,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.Flatten()
        )
        
        self.lip_fc = nn.Linear(32*48*12, FEATURE_NUM)

        
        self.head_cnn = nn.Sequential(
            nn.Conv2d(1,16,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.Conv2d(16,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.Flatten()
        )
       
        self.head_fc= nn.Linear(32*48*6, FEATURE_NUM)

       
        self.fusion_fc = nn.Sequential(
            nn.Linear(FEATURE_NUM * 9, FEATURE_NUM * 2),
            nn.ReLU(),
            nn.Linear(FEATURE_NUM * 2, FEATURE_NUM),
            nn.ReLU(),
            nn.Linear(FEATURE_NUM, action_dim)
        )
        
    def forward(self, abr_state, eye_data, lip_data, head_data):
    
      
        if abr_state.dim() == 4 and abr_state.size(1) == 1:
            abr_state = abr_state.squeeze(1)  
        
        B = abr_state.size(0)
        W = abr_state.size(2)  
        
       
        row0 = abr_state[:, 0:1, W-1:W]  
        row0 = row0.reshape(B, 1)
        row0_feat = F.relu(self.fc0(row0))
        
       
        row1 = abr_state[:, 1:2, W-1:W] 
        row1 = row1.reshape(B, 1)
        row1_feat = F.relu(self.fc1(row1))
        
    
        if W < 8:
            pad_size = 8 - W
            pad_tensor = torch.zeros(B, 1, pad_size, device=abr_state.device, dtype=abr_state.dtype)
            row2 = torch.cat([abr_state[:, 2:3, :], pad_tensor], dim=2)
        else:
            row2 = abr_state[:, 2:3, :8] 
        row2 = row2.reshape(B, 8)
        row2_feat = F.relu(self.fc2(row2))
        
    
        if W < 8:
            pad_size = 8 - W
            pad_tensor = torch.zeros(B, 1, pad_size, device=abr_state.device, dtype=abr_state.dtype)
            row3 = torch.cat([abr_state[:, 3:4, :], pad_tensor], dim=2)
        else:
            row3 = abr_state[:, 3:4, :8]
        row3 = row3.reshape(B, 8)
        row3_feat = F.relu(self.fc3(row3))
        

        int_action_dim = self.action_dim  # e.g. 6
        if W < int_action_dim:
          
            pad_cols = int_action_dim - W
            row4 = abr_state[:, 4:5, :W]
            pad_tensor = torch.zeros(B, 1, pad_cols, device=abr_state.device, dtype=abr_state.dtype)
            row4 = torch.cat([row4, pad_tensor], dim=2)
        else:
            row4 = abr_state[:, 4:5, :int_action_dim]
        row4 = row4.reshape(B, int_action_dim)
        row4_feat = F.relu(self.fc4(row4))

        row5 = abr_state[:, 5:6, W-1:W]  # shape: [B,1,1]
        row5 = row5.reshape(B, 1)
        row5_feat = F.relu(self.fc5(row5))

        abr_merged = torch.cat([row0_feat, row1_feat, row2_feat, row3_feat, row4_feat, row5_feat], dim=1)

        eye_c = self.eye_cnn(eye_data)
        eye_feat = F.relu(self.eye_fc(eye_c))
        
        lip_c = self.lip_cnn(lip_data)
        lip_feat = F.relu(self.lip_fc(lip_c))
        
        head_c = self.head_cnn(head_data)
        head_feat = F.relu(self.head_fc(head_c))
        
        # Merge all features.
        merged = torch.cat([abr_merged, eye_feat, lip_feat, head_feat], dim=1)
        q_values = self.fusion_fc(merged)
        return q_values



In [ ]:
# Cell: Load or Train RF and Batch‑Infer QoE for Eye_train
import os
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Paths
data3_dir     = "./Data3"
eye_train_dir = "./QoE_train/Eye_train"
preds_csv     = "./Preds/preds_RF_EyeTrain.csv"
os.makedirs(os.path.dirname(preds_csv), exist_ok=True)

def load_or_train_and_infer():
    # 1) If preds CSV exists, just load it
    if os.path.exists(preds_csv):
        df_preds = pd.read_csv(preds_csv)
        if {"UniqueID","PredictedQoE"}.issubset(df_preds.columns):
            print(f"[RF] Loaded existing {len(df_preds)} predictions from {preds_csv}")
            return df_preds
        else:
            print(f"[RF][Error] {preds_csv} missing required columns; retraining...")

    # 2) Train RF on Data3
    df_eye3 = load_eye_tracking_data(data3_dir)
    seg3    = process_segments(df_eye3, target_length=48)    # adjust target_length if needed
    bal3    = balance_segments(seg3)
    combo   = ("eye_data",)

    _, flat_dict3, label_dict3, _ = extract_modality_data(bal3, modalities, combo)
    df_flat3 = pd.DataFrame.from_dict(flat_dict3, orient="index")
    df_flat3["QoE"] = pd.Series(label_dict3)

    X3 = df_flat3.drop(columns=["QoE"]).apply(pd.to_numeric, errors="coerce").fillna(0)
    y3 = df_flat3["QoE"].astype(int)

    le = LabelEncoder()
    y3_enc = le.fit_transform(y3)
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X3, y3_enc)
    print(f"[RF] Trained on Data3: {X3.shape[0]} samples × {X3.shape[1]} features")

    # 3) Batch‑infer on Eye_train
    df_eye_t = load_eye_tracking_data(eye_train_dir)
    seg_t    = process_segments(df_eye_t, target_length=48)
    _, flat_dict_t, _, _ = extract_modality_data(seg_t, modalities, combo)

    results = []
    ids = list(flat_dict_t.keys())
    batch_size = 2000
    for i in range(0, len(ids), batch_size):
        batch_ids = ids[i:i+batch_size]
        batch_mat = [flat_dict_t[uid] for uid in batch_ids]
        Xb = pd.DataFrame(batch_mat).apply(pd.to_numeric, errors="coerce").fillna(0)
        preds_enc = rf.predict(Xb)
        preds = le.inverse_transform(preds_enc)
        results.extend(zip(batch_ids, preds))
        print(f"[RF] Inferred batch {i//batch_size+1} ({len(batch_ids)} segments)")

    # 4) Save CSV
    df_out = pd.DataFrame(results, columns=["UniqueID", "PredictedQoE"])
    df_out.to_csv(preds_csv, index=False)
    print(f"[RF] Saved {len(df_out)} predictions to {preds_csv}")
    return df_out

# Execute
df_predictions = load_or_train_and_infer()


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque
from tqdm import tqdm

# ── define your discrete action space ──────────────────────────────
bitrates = [2000, 6000, 9500, 15000, 30000, 85000]  # in kbps
bitrate_to_idx = {b: i for i, b in enumerate(bitrates)}



def debug_tensor_if_nan(tensor, tensor_name="tensor"):
    if torch.isnan(tensor).any():
        print(f"[WARNING] {tensor_name} contains NaN values.")

def debug_model_parameters(model):
    for name, param in model.named_parameters():
        if torch.isnan(param).any():
            print(f"[WARNING] Parameter '{name}' contains NaN values.")
        if param.grad is not None and torch.isnan(param.grad).any():
            print(f"[WARNING] Gradient of parameter '{name}' contains NaN values.")

def train_DqnBio_plus_bio(
    experiences,
    model,
    episodes=5,
    batch_size=16,
    gamma=0.99,
    lr=1e-4
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    from copy import deepcopy
    target_model = type(model)()
    target_model.load_state_dict(deepcopy(model.state_dict()))
    target_model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    replay_buffer = deque(experiences, maxlen=50000)
    print(f"[INFO] Loaded {len(replay_buffer)} experiences for training multi-branch DqnBio DQN.")

    update_target_every = 100  # episodes
    for ep in range(episodes):
        random.shuffle(replay_buffer)
        buffer_list = list(replay_buffer)
        for i in range(0, len(buffer_list), batch_size):
            batch = buffer_list[i:i+batch_size]


            if len(batch) < batch_size:
                break

            abr_s_list, eye_s_list, lip_s_list, head_s_list = [],[],[],[]
            act_list, rew_list, done_list = [],[],[]
            abr_sn_list, eye_sn_list, lip_sn_list, head_sn_list = [],[],[],[]

            for (abr_s, eye_s, lip_s, head_s,
                 a, r,
                 abr_sn, eye_sn, lip_sn, head_sn,
                 dn) in batch:
                abr_s_list.append(abr_s)
                eye_s_list.append(eye_s)
                lip_s_list.append(lip_s)
                head_s_list.append(head_s)
                act_list.append(a)
                rew_list.append(r)
                done_list.append(float(dn))

                if not dn:
                    abr_sn_list.append(abr_sn)
                    eye_sn_list.append(eye_sn)
                    lip_sn_list.append(lip_sn)
                    head_sn_list.append(head_sn)
                else:
                    abr_sn_list.append(np.zeros_like(abr_s))
                    eye_sn_list.append(np.zeros_like(eye_s))
                    lip_sn_list.append(np.zeros_like(lip_s))
                    head_sn_list.append(np.zeros_like(head_s))

            # Convert to tensors
            abr_s_t = torch.tensor(np.array(abr_s_list), dtype=torch.float32, device=device)   # (B,6,8)
            eye_s_t = torch.tensor(np.array(eye_s_list), dtype=torch.float32, device=device)
            if eye_s_t.ndim==3:
                eye_s_t= eye_s_t.unsqueeze(1)  # => (B,1,480,40)
            lip_s_t = torch.tensor(np.array(lip_s_list), dtype=torch.float32, device=device)
            if lip_s_t.ndim==3:
                lip_s_t= lip_s_t.unsqueeze(1)  # => (B,1,480,37)
            head_s_t= torch.tensor(np.array(head_s_list),dtype=torch.float32,device=device)
            if head_s_t.ndim==3:
                head_s_t= head_s_t.unsqueeze(1) # => (B,1,480,6)

            # map raw bitrate values → action indices 0–5
            try:
              act_idx = [bitrate_to_idx[int(a)] for a in act_list]
            except KeyError as e:
                act_idx = 3
            act_t    = torch.tensor(act_idx, dtype=torch.long, device=device)

            rew_t= torch.tensor(rew_list,dtype=torch.float32,device=device)
            done_t= torch.tensor(done_list,dtype=torch.float32,device=device)

            abr_sn_t= torch.tensor(np.array(abr_sn_list),dtype=torch.float32,device=device)
            eye_sn_t= torch.tensor(np.array(eye_sn_list),dtype=torch.float32,device=device)
            if eye_sn_t.ndim==3:
                eye_sn_t= eye_sn_t.unsqueeze(1)
            lip_sn_t= torch.tensor(np.array(lip_sn_list),dtype=torch.float32,device=device)
            if lip_sn_t.ndim==3:
                lip_sn_t= lip_sn_t.unsqueeze(1)
            head_sn_t= torch.tensor(np.array(head_sn_list),dtype=torch.float32,device=device)
            if head_sn_t.ndim==3:
                head_sn_t= head_sn_t.unsqueeze(1)

            qvals = model(abr_s_t, eye_s_t, lip_s_t, head_s_t)
            # if action > 6 find the index of that bitrate
            act_t = act_t.view(-1)
            if act_t.max() > 6:
                act_t = torch.tensor([bitrates.index(a) for a in act_t], dtype=torch.long, device=device)
            current_q = qvals.gather(1, act_t.unsqueeze(1)).squeeze(1)

            with torch.no_grad():
                qvals_next= target_model(abr_sn_t, eye_sn_t, lip_sn_t, head_sn_t)
                max_next_q= qvals_next.max(dim=1)[0]
                target_q = rew_t + gamma*(1.-done_t)*max_next_q

            loss = criterion(current_q, target_q)
            optimizer.zero_grad()
            loss.backward()
            debug_model_parameters(model)
            optimizer.step()

        if (ep+1)% update_target_every==0:
            target_model.load_state_dict(model.state_dict())
            print(f"[INFO] Episode {ep+1}: updated target network.")

    print("[INFO] Multi-branch DqnBio training complete.")
    return model


In [ ]:
# After you call load_or_train_and_infer():
df_preds = load_or_train_and_infer()

# Build baseline_preds dict:
baseline_preds = {"RF": {}}
for _, row in df_preds.iterrows():
    baseline_preds["RF"][row["UniqueID"]] = float(row["PredictedQoE"])

print(baseline_preds["RF"])
print(f"Total segments with preds: {len(baseline_preds['RF'])}")


In [ ]:
%pip install onnx

In [ ]:
import os
import numpy as np

import re

def parse_uid(uid):
    # uid == "video_<X>_segment_<Y>"
    m = re.match(r"video_(\d+)_segment_(\d+)", uid)
    if not m:
        raise ValueError(f"bad uid: {uid}")
    video_id, seg_id = map(int, m.groups())
    return video_id, seg_id

# … later …



def build_abr_state(net_data, chunk_idx, chunk_size_dict, buff =12.0):

    state_6x8 = np.zeros((6,8), dtype=np.float32)
    max_bitrate = 85000.0
    buffer_norm = buff
    chunk_norm = 1e6
    # Row 0: Bitrate normalized (insert in last column)
    bitrate = net_data.get("Bitrate", 0.0)
    state_6x8[0,-1] = bitrate / max_bitrate
    # Row 1: Buffer level normalized (insert in last column)
    buffer_val = net_data.get("BufferLevel", 0.0)
    state_6x8[1,-1] = buffer_val / buffer_norm
    # Row 2: Throughput history (fill entire row)
    thr_hist = net_data.get("ThroughputHistory", [])
    # Normalize throughput values to Mbps
    thr_hist = [val / 1000.0 for val in thr_hist]
    # normaliuze between 0 and  1
    thr_hist = [val / max_bitrate for val in thr_hist]
    row2 = np.zeros(8, dtype=np.float32)
    if thr_hist:
        L = len(thr_hist)
        if L >= 8:
            row2[:] = thr_hist[-8:]
        else:
            row2[-L:] = thr_hist
    state_6x8[2,:] = row2
   
    state_6x8[3,:] = 0.0
    # Row 4: Next chunk sizes (first 6 colum2,3,4,5,6ns; normalized from bytes to MB)
    next_sizes = np.zeros(6, dtype=np.float32)
    for b in range(6):
        seg_sizes = chunk_size_dict.get(b, [])
        if chunk_idx < len(seg_sizes):
            next_sizes[b] = seg_sizes[chunk_idx] / chunk_norm
    state_6x8[4, 0:6] = next_sizes
    state_6x8[5,-1] = 0.0
    return state_6x8

def load_chunk_sizes(chunk_dir="./chunkSizes"):
    
    ACTION_DIM = 6
    chunk_sizes = {}
    for b in range(ACTION_DIM):
        fname = os.path.join(chunk_dir, f"video_size_{b+1}")
        if not os.path.exists(fname):
            print(f"[Warning] File {fname} not found; using empty list for bitrate {b}.")
            chunk_sizes[b] = []
            continue
        with open(fname, "r") as f:
            lines = f.read().splitlines()
            sizes = [int(line.strip()) for line in lines if line.strip()]
        chunk_sizes[b] = sizes
    return chunk_sizes

# Load the chunk sizes once.
chunk_size_dict = load_chunk_sizes()
  

In [ ]:
# Remove any entries whose 'network' data is missing or empty
merged_dict = {
    uid: info
    for uid, info in merged_dict.items()
    if info.get("network")
}

# Now merged_dict only contains UIDs with non-empty network data
for uid, info in merged_dict.items():
    video_id, segment_id = parse_uid(uid)
    net_data = info["network"]
    
    abr_state = build_abr_state(net_data, segment_id, chunk_size_dict)


In [ ]:
import torch
import numpy as np
import os
import random


buffArr = [80]  
alpha_values = np.arange(0.6, 0.61, 0.2)  


trained_models = {}

all_experiences = None  

# Loop over each buffer and alpha combination.
for buff in buffArr:
    modified_dict = {}
    for uid, info in merged_dict.items():
        new_info = info.copy()
        net = info["network"].copy()
        # Override BufferLevel to be the minimum of the original and buff.
        
        net["BufferLevel"] = min(info["network"].get("BufferLevel", 0.0), buff)
        new_info["network"] = net
        modified_dict[uid] = new_info

    # Sort UIDs (using parse_uid defined earlier)
    sorted_uids = sorted(modified_dict.keys(), key=parse_uid)
    prev_action = None

    for alpha in alpha_values:
        experiences = []
        for i in range(len(sorted_uids)):
            uid = sorted_uids[i]
            info = modified_dict[uid]
            net_data = info["network"]
            chunk_idx = net_data.get("chunk_idx", 0)
            # Build the 6×8 ABR state using your row-slicing function.
            # --- before: abr_s = build_abr_state(...) ---
            raw = build_abr_state(net_data, chunk_idx, chunk_size_dict, buff)  # shape (6,8)
            
         

            
            abr_s = raw

            eye_s  = info["eye"]   
            lip_s  = info["lip"]    
            head_s = info["head"]   
            
            a = net_data.get("Bitrate", 0)
            
           
            subjective_qoe = baseline_preds['RF'][uid]
            r = calculate_reward(
                    selected_action=a,
                    throughput_history=net_data.get("ThroughputHistory", []),
                    buffer_level=net_data.get("BufferLevel", 0.0),
                    rebuffer_time=net_data.get("RebufferTime", 0.0),
                    subjective_qoe=subjective_qoe,
                    previous_action=prev_action,
                    alpha=alpha
                )
            done = info.get("done", False)
            prev_action = a

            # Build next state.
            if "next_network" in info:
                net_next = info["next_network"]
                chunk_idx_next = net_next.get("chunk_idx", 0)
                abr_sn = build_abr_state(net_next, chunk_idx_next, chunk_size_dict)
                eye_sn = info["next_eye"]
                lip_sn = info["next_lip"]
                head_sn = info["next_head"]
                done_flag = done
            else:
                if i < len(sorted_uids)-1:
                    uid_next = sorted_uids[i+1]
                    info_next = modified_dict[uid_next]
                    net_next = info_next["network"]
                    chunk_idx_next = net_next.get("chunk_idx", 0)
                    abr_sn = build_abr_state(net_next, chunk_idx_next, chunk_size_dict)
                    eye_sn = info_next["eye"]
                    lip_sn = info_next["lip"]
                    head_sn = info_next["head"]
                    video_id, _ = parse_uid(uid)
                    video_id_next, _ = parse_uid(uid_next)
                    done_flag = (video_id != video_id_next)
                else:
                    abr_sn = np.zeros_like(abr_s)
                    eye_sn = np.zeros_like(eye_s)
                    lip_sn = np.zeros_like(lip_s)
                    head_sn = np.zeros_like(head_s)
                    done_flag = True

            experiences.append((abr_s, eye_s, lip_s, head_s,
                                a, r,
                                abr_sn, eye_sn, lip_sn, head_sn,
                                done_flag))

        print(f"[INFO] For Buffer={buff} and Alpha={alpha:.1f}: Built {len(experiences)} experiences.")

        # Train the model for this configuration.
        model_instance = DQNBIO(action_dim=6)
        
        # model_instance.fusion_fc[-1].bias.data = torch.tensor([-0.95, 0.50, 0.32, 0.55, 0.32, 0.44])


        trained_model = train_DqnBio_plus_bio(
            experiences=experiences,
            model=model_instance,
            episodes=100,
            batch_size=32,
            gamma=0.99,
            lr=1e-4
        )

        # Export the trained model to ONNX.
        trained_model.eval()
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        trained_model.to(device)

        dummy_abr  = torch.randn(1,1,6,8, dtype=torch.float32, device=device)
        dummy_eye  = torch.randn(1,1,48,26, dtype=torch.float32, device=device)
        dummy_lip  = torch.randn(1,1,48,12, dtype=torch.float32, device=device)
        dummy_head = torch.randn(1,1,48,6,  dtype=torch.float32, device=device)

        onnx_filename = f"DqnBio_plus_bio_dqn_alpha_{alpha:.1f}_buff_{buff}.onnx"
        torch.onnx.export(
            trained_model,
            (dummy_abr, dummy_eye, dummy_lip, dummy_head),
            onnx_filename,
            input_names=["abr_state", "eye", "lip", "head"],
            output_names=["q_values"],
            opset_version=11
        )
        print(f"[INFO] Exported model for Buffer {buff} and Alpha {alpha:.1f} to {onnx_filename}")
        trained_models[(buff, alpha)] = trained_model

print("\n[INFO] Training complete for all configurations.")
for key, model in trained_models.items():
    print(f"Buffer {key[0]}, Alpha {key[1]:.1f}")


In [ ]:
#!/usr/bin/env python3
import os
import numpy as np
import onnxruntime as ort
import random

# ----- Configuration -----
# Paths to the two ONNX models:
MODEL_PATH_MULTI = "DqnBio_plus_bio_dqn_alpha_0.6_buff_80.onnx"  # Multi-branch: expects inputs ["abr_state", "eye", "lip", "head80
MODEL_PATH_NET   = "DqnBio_dqn.onnx"            # Network-only: expects input "abr_state"
CHUNK_DIR = "./chunkSizes"   # Files: "video_size_0", ..., "video_size_5"
MAX_BITRATE = 85.0         # in kbps
BUFFER_NORM = 12.0            # normal# Bias the final layer to prefer actions 3 > 5 > 1 > othersize buffer level (max buffer = 4 sec)
THR_NORM = 85               # normalize throughput (max expected throughput in Mbps)
CHUNK_NORM = 1e6             # to convert bytes to MB
NUM_COLUMNS = 8              # state has 8 columns per row
NUM_ROWS = 6                 # state has 6 rows
ACTION_DIM = 6               # number of actions
NUM_ITER = 5000               # total simulation iterations
bitrates = [2000, 6000, 9500, 15000, 30000, 85000]

# ----- Load chunk sizes -----
def load_chunk_sizes(chunk_dir=CHUNK_DIR):
    chunk_sizes = {}
    for b in range(ACTION_DIM):
        fname = os.path.join(chunk_dir, f"video_size_{b+1}")
        if not os.path.exists(fname):
            print(f"[Warning] File {fname} not found; using empty list for bitrate {b}.")
            chunk_sizes[b] = []
            continue
        with open(fname, "r") as f:
            lines = f.read().splitlines()
            sizes = [int(line.strip()) for line in lines if line.strip()]
        chunk_sizes[b] = sizes
    return chunk_sizes

chunk_size_dict = load_chunk_sizes()
class StateManager:
    def __init__(self, num_rows=NUM_ROWS, num_cols=NUM_COLUMNS, action_dim=ACTION_DIM):
        self.state = np.zeros((num_rows, num_cols), dtype=np.float32)
        self.action_dim = action_dim

    def update_state(self, net_metrics, next_seg_idx, chunk_size_dict):
        # Roll rows 0,1,2,3,5 left by one column.
        self.state[0, :] = np.roll(self.state[0, :], -1)
        self.state[1, :] = np.roll(self.state[1, :], -1)
        self.state[2, :] = np.roll(self.state[2, :], -1)
        self.state[3, :] = np.roll(self.state[3, :], -1)
        self.state[5, :] = np.roll(self.state[5, :], -1)
        # Roll row 4 as well.
        self.state[4, :] = np.roll(self.state[4, :], -1)
        bitrate = net_metrics.get("Bitrate", 0)
        self.state[0, -1] = bitrate / MAX_BITRATE
        buffer_val = net_metrics.get("BufferLevel", 0.0)
        self.state[1, -1] = buffer_val / BUFFER_NORM
        thr_hist = net_metrics.get("ThroughputHistory", [])
        new_thr = (thr_hist[-1] ) if thr_hist else 0.0
        self.state[2, -1] = new_thr
        self.state[3, -1] = 0.0
        next_sizes = np.zeros(self.action_dim, dtype=np.float32)
        for b in range(self.action_dim):
            seg_list = chunk_size_dict.get(b, [])
            if next_seg_idx < len(seg_list):
                next_sizes[b] = seg_list[next_seg_idx] / CHUNK_NORM
            else:
                next_sizes[b] = 0.0
        self.state[4, 0:self.action_dim] = next_sizes
        self.state[5, -1] = 0.0

        return self.state.copy()
prev_rate = 2000
def simulate_network_metrics():

    bandwidth = random.uniform(10000, 85000)
    throughput_history = [bandwidth +random.uniform(-0.05, 0.05) for _ in range(NUM_COLUMNS)]
    buffer_level = random.uniform(0, 12)
    rebuffer_time = 0.0
    bitrate = prev_rate
    return {
        "BufferLevel": buffer_level,
        "ThroughputHistory": throughput_history,
        "RebufferTime": rebuffer_time,
        "Bitrate": bitrate
    }

session_multi = ort.InferenceSession(MODEL_PATH_MULTI)
input_names_multi = [inp.name for inp in session_multi.get_inputs()]
print(f"[INFO] Multi-branch ONNX model loaded. Input names: {input_names_multi}")
state_manager = StateManager(num_rows=NUM_ROWS, num_cols=NUM_COLUMNS, action_dim=ACTION_DIM)
counts = [0, 0, 0, 0, 0, 0]
unique_actions_multi = set()

count=0
for seg in range(NUM_ITER):
    net_metrics = simulate_network_metrics()
    next_seg_idx = (seg + 1) % 16  # wrap-around every 16 segments
    state = state_manager.update_state(net_metrics, next_seg_idx, chunk_size_dict)
    
    input_tensor = state[np.newaxis, np.newaxis, :, :].astype(np.float32)  # shape (1,1,6,8)


    dummy_eye = np.random.rand(1, 1, 48, 26).astype(np.float32)
    dummy_lip = np.random.rand(1, 1, 48, 12).astype(np.float32)
    dummy_head = np.random.rand(1, 1, 48, 6).astype(np.float32)

    outputs_multi = session_multi.run(None, {
        input_names_multi[0]: input_tensor,
        input_names_multi[1]: dummy_eye,
        input_names_multi[2]: dummy_lip,
        input_names_multi[3]: dummy_head
    })
    countsEach = {}
    q_values_multi = outputs_multi[0]
    action_multi = int(np.argmax(q_values_multi, axis=1)[0])
    unique_actions_multi.add(action_multi) 
    
    if count == 0:
      actions_counts = [0,0,0,0,0,0]
      count=1
    else:
        pass
    actions_counts[action_multi] += 1
    print(f"Segment {seg+1:02d}:") 
    print("  Simulated Network Metrics:")
    print("    Bitrate (kbps):", net_metrics["Bitrate"])
    print("    Buffer Level (s):", net_metrics["BufferLevel"])
    print("    Throughput History (Mbps):", net_metrics["ThroughputHistory"])
    print("  6×8 ABR State (normalized & rolled):\n", state)
    print("  Q-values:", q_values_multi)
    print("  Chosen action (bitrate index):", action_multi)
    print("  Corresponding bitrate (kbps):", bitrates[action_multi])
    print("  Actions counts:", actions_counts)
    print("-" * 60)
    prev_rate = bitrates[action_multi]
    counts[action_multi] += 1
    print(countsEach)
    
    

    # Run inference for network-only model (only one input, the ABR state).
    # outputs_net = session_net.run(None, {input_names_net[0]: input_tensor})
    # q_values_net = outputs_net[0]
    # action_net = int(np.argmax(q_values_net axis=1)[0])
    # unique_actions_net.add(action_net)

print("\nMulti-branch model unique actions:", unique_actions_multi)
print("Number of unique actions (multi-branch):", len(unique_actions_multi))
print("Actions counts:", actions_counts)


In [ ]:
# clear gpu cache
torch.cuda.empty_cache()
